# A stud wall: from lines to a fabrication file

Companion to [Week 06 §11](../README.md#11-graphs-in-architecture-three-applications)
and `Lecture_06/graph_applications/timber_frame.py`, which does all of this
in two function calls. Here every stage stops and shows what it holds:

**centrelines → roles → beams → how they meet → a rule → joints → BTLx**

> **Needs the timber extra**: `uv run --extra timber jupyter lab`. Units
> are millimetres. Do `07_timber_beams_and_joints` first.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from collections import Counter

from compas.colors import Color
from compas.datastructures import Graph
from compas.geometry import Line
from compas.tolerance import Tolerance
from compas_timber.connections import JointTopology, LButtJoint, TButtJoint
from compas_timber.elements import Beam
from compas_timber.model import TimberModel

## Step 1 — the wall is a function that returns lines
Plates top and bottom, a stud every `spacing`, a window (studs that would
cross it become short *cripple* studs), a brace. All the design is in this
one function; everything after it is derived.

In [ ]:
def stud_wall(length=3600, height=2400, spacing=600, opening=(1200, 3000, 900, 2000), brace=True):
    """{name: Line}. opening = (x from, x to, sill height, header height)."""
    x0, x1, sill, head = opening
    lines = {
        "bottom_plate": Line([0, 0, 0], [length, 0, 0]),
        "top_plate": Line([0, 0, height], [length, 0, height]),
        "sill": Line([x0, 0, sill], [x1, 0, sill]),
        "header": Line([x0, 0, head], [x1, 0, head]),
    }
    for i, x in enumerate(range(0, length + 1, spacing)):
        if x0 < x < x1:
            lines[f"cripple_{i}_low"] = Line([x, 0, 0], [x, 0, sill])
            lines[f"cripple_{i}_high"] = Line([x, 0, head], [x, 0, height])
        else:
            lines[f"stud_{i}"] = Line([x, 0, 0], [x, 0, height])
    if brace:
        lines["brace"] = Line([0, 0, 0], [x0, 0, height])
    return lines


def elevation(lines, title, size=(640, 440)):
    """The wall seen from the front: x across, z up."""
    return draw_points([], title, lines=[((l.start.x, l.start.z), (l.end.x, l.end.z)) for l in lines.values()],
                       size=size, bounds=[(0, 0), (3600, 2400)])


lines = stud_wall()
print(len(lines), "centrelines:", ", ".join(lines))
elevation(lines, "the stud wall as centrelines")

## Step 2 — say what each line *is*, before it becomes a beam
A plate, a stud, part of the opening, a brace. This is the first step of
the topology-first workflow used in robotic timber research: decide what
each member **is**, then let rules decide how members meet.

In [ ]:
def category(name):
    for prefix, role in (("bottom_plate", "plate"), ("top_plate", "plate"), ("stud", "stud"),
                         ("cripple", "stud"), ("sill", "opening"), ("header", "opening"), ("brace", "brace")):
        if name.startswith(prefix):
            return role
    return "other"


print(Counter(category(name) for name in lines))

## Step 3 — beams, and the model
Every line gets the same 60 × 120 section. The role travels along as an
attribute, so later stages can ask for it.

In [ ]:
model = TimberModel(tolerance=Tolerance(unit="MM", absolute=1e-3))
for name, line in lines.items():
    beam = Beam.from_centerline(line, 60, 120, z_vector=[0, 1, 0])
    beam.name = name
    beam.attributes["category"] = category(name)
    model.add_element(beam)

timber = sum(beam.length for beam in model.beams) / 1000
print(len(list(model.beams)), "beams,", f"{timber:.1f} m of 60 x 120")
view3d(*[(beam.blank, {"color": Color.from_hex("#e8590c" if beam.attributes["category"] == "brace" else "#f1c27d")})
         for beam in model.beams])

## Step 4 — how do the beams meet?
One call finds every pair that touches and classifies it. Count the
answers two ways: by topology, and by which roles are meeting.

In [ ]:
NAMES = {JointTopology.TOPO_L: "L", JointTopology.TOPO_T: "T", JointTopology.TOPO_X: "X"}

model.connect_adjacent_beams()
# sorted by name, so that the order (and every figure below) is the same on each run
candidates = sorted(model.joint_candidates, key=lambda c: sorted(e.name for e in c.elements))
print(len(candidates), "places where two beams meet")
print("by topology:", dict(Counter(NAMES[c.topology] for c in candidates)))
print("by roles:   ", dict(Counter(" - ".join(sorted(e.attributes["category"] for e in c.elements))
                                   for c in candidates)))
print("model graph:", model.graph.number_of_nodes(), "nodes,", model.graph.number_of_edges(), "edges")

### The same thing as a picture
The model graph has no coordinates of its own, so copy it into a plain
`Graph` with each beam at the middle of its centreline (the brace a little
lower, or it would hide the stud it crosses). Edge colour is the topology:
grey T, blue L, orange X.

In [ ]:
def element_graph(model):
    graph = Graph()
    for beam in model.beams:
        mid = beam.centerline.point_at(0.3 if beam.attributes["category"] == "brace" else 0.5)
        graph.add_node(beam.name, x=mid.x, y=mid.z, z=0.0, category=beam.attributes["category"])
    return graph


picture = element_graph(model)
shade = {"L": "#1c7ed6", "T": "#adb5bd", "X": "#e8590c"}
colours = {}
for c in candidates:
    a, b = [e.name for e in c.elements]
    picture.add_edge(a, b)
    colours[a, b] = shade[NAMES[c.topology]]
row(elevation(lines, "centrelines", size=(470, 400)),
    draw_graph(picture, "beams are nodes, meetings are edges", edge_colors=colours, size=(560, 400)))

## Step 5 — the rule
The design decision of this wall fits in a dictionary: *which joint for
which way of meeting.* X crossings would be lap joints, which need Rhino
(see `07_timber_beams_and_joints`), so the rule leaves them out and the code
**reports** them instead of failing.

In [ ]:
JOINT_RULE = {
    JointTopology.TOPO_L: LButtJoint,
    JointTopology.TOPO_T: TButtJoint,
}


def ends_at(a, b, tol=1.0):
    """True if an end of beam a lies on beam b: a butts into b."""
    for end in (a.centerline.start, a.centerline.end):
        if b.centerline.closest_point(end).distance_to_point(end) < tol + max(b.width, b.height) / 2:
            return True
    return False


made, left_open = Counter(), []
for candidate in candidates:
    joint = JOINT_RULE.get(candidate.topology)
    a, b = candidate.elements
    if joint is None:
        left_open.append((a.name, b.name, NAMES[candidate.topology]))
    elif joint is TButtJoint:
        main, cross = (a, b) if ends_at(a, b) else (b, a)      # the T joint's order, checked
        joint.create(model, main, cross)
        made[joint.__name__] += 1
    else:
        joint.create(model, a, b)
        made[joint.__name__] += 1

errors = model.process_joinery()
print("joints made:   ", dict(made))
print("left for Rhino:", left_open)
print("joinery errors:", len(errors))

## Step 6 — ask the graph
With the joints in place, questions about the wall are graph questions.
*Which beam has the most joints?* is a degree count. *Which beam is tied in
least?* is the smallest degree.

In [ ]:
degree = {beam.name: len(model.get_joints_for_element(beam)) for beam in model.beams}
for name in sorted(degree, key=degree.get, reverse=True)[:4]:
    print(f"{name:14s} {degree[name]} joints")
print("fewest:", min(degree, key=degree.get), "with", min(degree.values()))
top = max(degree.values())
draw_graph(picture, "joints per beam: orange = most", edge_colors=colours,
           node_colors={name: ramp(d / top) for name, d in degree.items()}, size=(640, 420))

## Step 7 — the fabrication file
`BTLxWriter` turns the model into BTLx. The count below is read back from
the XML that was just produced, not from what we meant to put in it.

In [ ]:
import xml.etree.ElementTree as ET

from compas_timber.fabrication import BTLxWriter

xml = BTLxWriter().model_to_xml(model)
parts, machining = 0, Counter()
for element in ET.fromstring(xml).iter():
    tag = element.tag.split("}")[-1]
    parts += tag == "Part"
    if tag == "Processings":
        machining.update(child.tag.split("}")[-1] for child in element)
print(f"{len(xml) / 1000:.0f} kB of BTLx: {parts} parts, machining {dict(machining)}")
order = sum(beam.blank_length for beam in model.beams) / 1000
print(f"timber to order: {order:.2f} m (centrelines were {timber:.2f} m)")

`timber_frame.py` writes this string to `output/stud_wall.btlx`. A notebook
is for looking; the script is the thing you would run in a pipeline.

## Step 8 — change the design, rerun the pipeline
Everything above hangs off `stud_wall()`. Wrap the stages in one function
and a design variant costs one line.

In [ ]:
def build(**design):
    model = TimberModel(tolerance=Tolerance(unit="MM", absolute=1e-3))
    for name, line in stud_wall(**design).items():
        beam = Beam.from_centerline(line, 60, 120, z_vector=[0, 1, 0])
        beam.name = name
        model.add_element(beam)
    model.connect_adjacent_beams()
    joints, crossings = 0, 0
    for candidate in list(model.joint_candidates):
        joint = JOINT_RULE.get(candidate.topology)
        a, b = candidate.elements
        if joint is None:
            crossings += 1
            continue
        if joint is TButtJoint and not ends_at(a, b):
            a, b = b, a
        joint.create(model, a, b)
        joints += 1
    model.process_joinery()
    metres = sum(beam.blank_length for beam in model.beams) / 1000
    return len(list(model.beams)), joints, crossings, metres


print(f"{'variant':28s} {'beams':>5s} {'joints':>6s} {'X left':>6s} {'timber':>8s}")
for label, design in (("as designed", {}),
                      ("studs at 400", {"spacing": 400}),
                      ("no brace", {"brace": False}),
                      ("wide window", {"opening": (600, 3000, 900, 2000)})):
    beams, joints, crossings, metres = build(**design)
    print(f"{label:28s} {beams:5d} {joints:6d} {crossings:6d} {metres:7.1f}m")

## Your turn
1. Change the rule so that plates and studs meet in a
   `TTenonMortiseJoint`. Which numbers in Step 7 change?
2. The brace crosses a stud (the X). Redesign `stud_wall` so that the brace
   ends on the stud instead. What topology is it now, and is anything
   still left for Rhino?
3. Find every beam that could be removed without the model graph falling
   apart (hint: `02_floor_plan`, critical doors).
4. Add a door: an opening whose sill height is 0. What does `stud_wall`
   produce, and what should it produce?